In [ ]:
import os
import json
import pickle
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, roc_auc_score
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import shap


In [ ]:

plt.style.use('dark_background')
sns.set_theme(style='darkgrid')

NEON_RED    = '#ff003c'
NEON_BLUE   = '#00d4ff'
NEON_GREEN  = '#00ff88'
NEON_ORANGE = '#ff8c00'

print(f"NumPy     : {np.__version__}")
print(f"Pandas    : {pd.__version__}")
print(f"PyTorch   : {torch.__version__}")
print(f"SHAP      : {shap.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
print("\n✅ All imports successful!")

In [ ]:
# ── Load Real Dataset ─────────────────────────────────────
DATA_PATH = r"C:\Users\Gagandeep Singh\Desktop\cyberdefense-platform\notebooks\data\data_file.csv"

df = pd.read_csv(DATA_PATH)

print(f"✅ Dataset loaded successfully!")
print(f"   Shape     : {df.shape}")
print(f"   Rows      : {df.shape[0]:,}")
print(f"   Columns   : {df.shape[1]}")
print(f"\n── Column Names ──────────────────────────────")
print(df.columns.tolist())
print(f"\n── First 3 Rows ──────────────────────────────")
df.head(3)

In [ ]:
# ── Check label distribution ──────────────────────────────
print("── Label Distribution ────────────────────────")
print(df['Benign'].value_counts())
print(f"\n  0 = Ransomware : {(df['Benign']==0).sum():,}")
print(f"  1 = Benign      : {(df['Benign']==1).sum():,}")

# ── Check for missing values ──────────────────────────────
print(f"\n── Missing Values ────────────────────────────")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "  ✅ No missing values!")

# ── Drop non-numeric columns ──────────────────────────────
df = df.drop(columns=['FileName', 'md5Hash'], errors='ignore')
print(f"\n── After dropping text columns ───────────────")
print(f"   Shape: {df.shape}")
print(f"   Features: {[c for c in df.columns if c != 'Benign']}")

# ── Basic stats ───────────────────────────────────────────
print(f"\n── Data Types ────────────────────────────────")
print(df.dtypes)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('🛡️ Dataset Overview', fontsize=16, color=NEON_BLUE, fontweight='bold')

# ── Plot 1: Class Balance Bar Chart ───────────────────────
counts = df['Benign'].value_counts()
bars = axes[0].bar(['Ransomware (0)', 'Benign (1)'], counts.values,
                   color=[NEON_RED, NEON_GREEN], edgecolor='white', linewidth=0.8)
axes[0].set_title('Class Distribution', color=NEON_BLUE)
axes[0].set_ylabel('Count', color='white')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', color='white', fontweight='bold')

# ── Plot 2: BitcoinAddresses Distribution ─────────────────
axes[1].hist(df[df['Benign']==0]['BitcoinAddresses'], bins=30,
             color=NEON_RED, alpha=0.7, label='Ransomware')
axes[1].hist(df[df['Benign']==1]['BitcoinAddresses'], bins=30,
             color=NEON_GREEN, alpha=0.7, label='Benign')
axes[1].set_title('Bitcoin Addresses Distribution', color=NEON_BLUE)
axes[1].set_xlabel('Bitcoin Addresses Count', color='white')
axes[1].legend()

# ── Plot 3: NumberOfSections Distribution ─────────────────
axes[2].hist(df[df['Benign']==0]['NumberOfSections'], bins=20,
             color=NEON_RED, alpha=0.7, label='Ransomware')
axes[2].hist(df[df['Benign']==1]['NumberOfSections'], bins=20,
             color=NEON_GREEN, alpha=0.7, label='Benign')
axes[2].set_title('Number of PE Sections', color=NEON_BLUE)
axes[2].set_xlabel('Sections Count', color='white')
axes[2].legend()

plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved as dataset_overview.png")

In [ ]:
FEATURES = ['Machine', 'DebugSize', 'DebugRVA', 'MajorImageVersion',
            'MajorOSVersion', 'ExportRVA', 'ExportSize', 'IatVRA',
            'MajorLinkerVersion', 'MinorLinkerVersion', 'NumberOfSections',
            'SizeOfStackReserve', 'DllCharacteristics', 'ResourceSize',
            'BitcoinAddresses']
LABEL = 'Benign'

plt.figure(figsize=(14, 10))
corr = df[FEATURES + [LABEL]].corr()
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 7},
            cbar_kws={'shrink': 0.8})
plt.title('🔥 Feature Correlation Heatmap', color=NEON_BLUE,
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Heatmap saved!")
print(f"\n── Top correlations with label ───────────────")
print(corr[LABEL].drop(LABEL).sort_values(ascending=False).to_string())

In [ ]:
# ── Define Features & Label ───────────────────────────────
FEATURES = ['Machine', 'DebugSize', 'DebugRVA', 'MajorImageVersion',
            'MajorOSVersion', 'ExportRVA', 'ExportSize', 'IatVRA',
            'MajorLinkerVersion', 'MinorLinkerVersion', 'NumberOfSections',
            'SizeOfStackReserve', 'DllCharacteristics', 'ResourceSize',
            'BitcoinAddresses']
LABEL = 'Benign'

X = df[FEATURES].values
y = df[LABEL].values

# ── Train/Test Split ──────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Scale Features ────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ── Save Scaler ───────────────────────────────────────────
MODELS_DIR = r"C:\Users\Gagandeep Singh\Desktop\cyberdefense-platform\backend\models"
os.makedirs(MODELS_DIR, exist_ok=True)

with open(os.path.join(MODELS_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

print(f"✅ Data prepared!")
print(f"   Training samples : {X_train.shape[0]:,}")
print(f"   Testing samples  : {X_test.shape[0]:,}")
print(f"   Features         : {X_train.shape[1]}")
print(f"   Scaler saved     : backend/models/scaler.pkl")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🛡️ Random Forest Results', fontsize=15, color=NEON_BLUE, fontweight='bold')

# ── Plot 1: Confusion Matrix ──────────────────────────────
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Ransomware', 'Benign'],
            yticklabels=['Ransomware', 'Benign'],
            ax=axes[0], linewidths=1)
axes[0].set_title('Confusion Matrix', color=NEON_BLUE)
axes[0].set_ylabel('Actual', color='white')
axes[0].set_xlabel('Predicted', color='white')

# ── Plot 2: Feature Importance ────────────────────────────
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
sorted_features = [FEATURES[i] for i in indices]
sorted_importances = importances[indices]

bars = axes[1].barh(sorted_features[::-1], sorted_importances[::-1],
                    color=NEON_BLUE, edgecolor='white', linewidth=0.5)
axes[1].set_title('Feature Importance', color=NEON_BLUE)
axes[1].set_xlabel('Importance Score', color='white')

plt.tight_layout()
plt.savefig('rf_results.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Save feature importances as JSON (used by backend) ────
importance_dict = {FEATURES[i]: float(importances[i]) for i in range(len(FEATURES))}
with open(os.path.join(MODELS_DIR, 'feature_importances.json'), 'w') as f:
    json.dump(importance_dict, f, indent=2)

print("✅ rf_results.png saved!")
print("✅ feature_importances.json saved!")
print(f"\n── Top 5 Most Important Features ────────────")
for i in range(5):
    print(f"  {i+1}. {sorted_features[i]:<22} {sorted_importances[i]:.4f}")

In [ ]:
# ── Convert to PyTorch Tensors ────────────────────────────
X_train_t = torch.FloatTensor(X_train_scaled)
X_test_t  = torch.FloatTensor(X_test_scaled)
y_train_t = torch.FloatTensor(y_train)
y_test_t  = torch.FloatTensor(y_test)

train_ds = TensorDataset(X_train_t, y_train_t)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)

# ── Define Neural Network ─────────────────────────────────
class RansomwareNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x).squeeze()

model = RansomwareNet(input_dim=len(FEATURES))
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)
print(f"\n✅ Model created!")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ── Train ─────────────────────────────────────────────────
EPOCHS = 30
train_losses = []

print(f"\n⏳ Training for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for xb, yb in train_dl:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_dl)
    train_losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:02d}/{EPOCHS} | Loss: {avg_loss:.4f}")

print("\n✅ Training complete!")

# ── Evaluate ──────────────────────────────────────────────
model.eval()
with torch.no_grad():
    dl_probs = model(X_test_t).numpy()
    dl_preds = (dl_probs >= 0.5).astype(int)

print(f"\n── Deep Learning Results ─────────────────────")
print(f"  Accuracy  : {accuracy_score(y_test, dl_preds):.4f}")
print(f"  Precision : {precision_score(y_test, dl_preds):.4f}")
print(f"  Recall    : {recall_score(y_test, dl_preds):.4f}")
print(f"  F1 Score  : {f1_score(y_test, dl_preds):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test, dl_probs):.4f}")

# ── Save Model ────────────────────────────────────────────
torch.save(model.state_dict(), os.path.join(MODELS_DIR, 'dl_model.pth'))

# Save architecture info for backend
model_info = {'input_dim': len(FEATURES), 'features': FEATURES}
with open(os.path.join(MODELS_DIR, 'dl_model_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)

print("✅ dl_model.pth saved!")
print("✅ dl_model_info.json saved!")

In [ ]:
print("⏳ Computing SHAP values (may take ~1 min)...")

# Use a sample of 500 rows for speed
sample_idx = np.random.choice(len(X_test), 500, replace=False)
X_sample = X_test[sample_idx]

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_sample)

# shap_values is a list [class0, class1] — we use class1 (Benign=1)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

print("✅ SHAP values computed!")

# ── Plot 1: Summary Bar Plot ──────────────────────────────
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_sample, feature_names=FEATURES,
                  plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Mean Impact)', color=NEON_BLUE)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ shap_bar.png saved!")

In [ ]:
# ── Plot 2: SHAP Dot Plot (shows direction of impact) ─────
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_sample, feature_names=FEATURES, show=False)
plt.title('SHAP Summary — Feature Impact Direction', color=NEON_BLUE)
plt.tight_layout()
plt.savefig('shap_dot.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ shap_dot.png saved!")

# ── Save mean SHAP values as JSON for backend/frontend ────
mean_shap = np.abs(sv).mean(axis=0)
shap_dict = {FEATURES[i]: float(mean_shap[i]) for i in range(len(FEATURES))}
shap_sorted = dict(sorted(shap_dict.items(), key=lambda x: x[1], reverse=True))

with open(os.path.join(MODELS_DIR, 'shap_values.json'), 'w') as f:
    json.dump(shap_sorted, f, indent=2)

print("✅ shap_values.json saved!")
print(f"\n── Top 5 Features by SHAP Impact ─────────────")
for i, (feat, val) in enumerate(list(shap_sorted.items())[:5]):
    print(f"  {i+1}. {feat:<22} {val:.4f}")

In [ ]:
# ── Verify all model files saved correctly ────────────────
print("── Model Files Verification ──────────────────")
expected_files = [
    'rf_model.pkl',
    'dl_model.pth',
    'scaler.pkl',
    'feature_importances.json',
    'dl_model_info.json',
    'shap_values.json'
]

all_ok = True
for fname in expected_files:
    fpath = os.path.join(MODELS_DIR, fname)
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) if exists else 0
    status = "✅" if exists else "❌"
    print(f"  {status} {fname:<30} {size/1024:.1f} KB")
    if not exists:
        all_ok = False

print(f"\n── Final Model Scorecard ─────────────────────")
print(f"  Random Forest")
print(f"    Accuracy  : 99.62%")
print(f"    ROC-AUC   : 99.98%")
print(f"    Top Feature: DllCharacteristics")
print(f"\n  Deep Neural Network (PyTorch)")
print(f"    Accuracy  : 98.30%")
print(f"    ROC-AUC   : 99.71%")
print(f"    Parameters: 12,801")
print(f"\n  Dataset")
print(f"    Total samples : 62,485")
print(f"    Ransomware    : 35,367")
print(f"    Benign        : 27,118")
print(f"    Features used : 15 PE file features")

if all_ok:
    print(f"\n✅ ALL FILES SAVED — Notebook complete!")
    print(f"   Models ready for Flask backend integration.")
else:
    print(f"\n⚠️  Some files missing — check errors above.")

In [ ]:
# ── Recreate feature_importances.json ─────────────────────
importances = rf.feature_importances_
importance_dict = {FEATURES[i]: float(importances[i]) for i in range(len(FEATURES))}

with open(os.path.join(MODELS_DIR, 'feature_importances.json'), 'w') as f:
    json.dump(importance_dict, f, indent=2)

print("✅ feature_importances.json saved!")

# ── Verify now all 6 files exist ──────────────────────────
print("\n── Final Check ───────────────────────────────")
for fname in expected_files:
    fpath = os.path.join(MODELS_DIR, fname)
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) if exists else 0
    status = "✅" if exists else "❌"
    print(f"  {status} {fname:<30} {size/1024:.1f} KB")